# Is Spark Still Relevant in 2026? DuckDB Was 7x Faster. Amazon's Data Complicates It.

Companion notebook for the article. Run all cells to reproduce every number. Needs PySpark 4 (with Java 17 or newer), DuckDB, Polars, PyArrow, pandas and scikit-learn. The setup cell writes 1.2 GB of synthetic Parquet next to the notebook on first run, and timings will differ from the article's two-core VM.

From [*Math for ML Engineers: From Backprop to RLHF*](https://mikaelyemane.github.io/ml-math-code/) by Mikael Yemane.

In [1]:
%pip install -q 'pyspark>=4.0' duckdb polars pyarrow pandas scikit-learn

## Setup

In [2]:
import os, sys
from pathlib import Path
import numpy as np, pyarrow as pa, pyarrow.parquet as pq
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # workers use this interpreter
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")       # single machine: skip the hostname lookup
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--conf spark.ui.showConsoleProgress=false pyspark-shell")
if not Path("users.parquet").exists():                     # 30 days x 1M events + 1M users, about 1.2 GB
    rng = np.random.default_rng(0)
    for d in range(30):
        n = 1_000_000
        Path(f"events/day={d:02d}").mkdir(parents=True, exist_ok=True)
        pq.write_table(pa.table({
            "user_id": rng.integers(0, 1_000_000, n),
            "event_type": rng.choice(np.array(["view", "click", "cart", "buy"]), n, p=[.7, .2, .07, .03]),
            "amount": np.round(rng.gamma(2.0, 20.0, n), 2),
            "latency_ms": rng.gamma(2.0, 150.0, n),
            "device": rng.choice(np.array(["ios", "android", "web"]), n),
            "session_id": rng.integers(0, 10**12, n),
            "page": rng.integers(0, 5000, n),
            "payload": rng.integers(0, 10**15, n).astype(str),
        }), f"events/day={d:02d}/part-0.parquet")
    pq.write_table(pa.table({
        "user_id": np.arange(1_000_000),
        "country": rng.choice(np.array(["US", "DE", "IN", "BR", "JP", "GB", "FR"]), 1_000_000),
        "signup_year": rng.integers(2015, 2026, 1_000_000),
    }), "users.parquet")

## The case that Spark still matters

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.master("local[*]").getOrCreate()
events = spark.read.parquet("events")       # 30M rows, 8 columns, partitioned by day (0-29)
users = spark.read.parquet("users.parquet")  # 1M rows

revenue = (events.join(users, "user_id")
           .filter((F.col("day") >= 23) & (F.col("event_type") == "buy") & (F.col("country") == "DE"))
           .groupBy("device")
           .agg(F.sum("amount").alias("revenue"), F.count("*").alias("orders")))
revenue.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Project (9)
            +- BroadcastHashJoin Inner BuildRight (8)
               :- Project (3)
               :  +- Filter (2)
               :     +- Scan parquet  (1)
               +- BroadcastExchange (7)
                  +- Project (6)
                     +- Filter (5)
                        +- Scan parquet  (4)


(1) Scan parquet 
Output [5]: [user_id#0L, event_type#1, amount#2, device#4, day#8]
Batched: true
Location: InMemoryFileIndex [file:/content/events]
PartitionFilters: [isnotnull(day#8), (day#8 >= 23)]
PushedFilters: [IsNotNull(event_type), EqualTo(event_type,buy), IsNotNull(user_id)]
ReadSchema: struct<user_id:bigint,event_type:string,amount:double,device:string>

(2) Filter
Input [5]: [user_id#0L, event_type#1, amount#2, device#4, day#8]
Condition : ((isnotnull(event_type#1) AND (event_type#1 = buy)) AND isnotnull(user_id#0L))

(3) Project
Out

In [4]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

def fit_one(pdf: pd.DataFrame) -> pd.DataFrame:
    X = pdf[["latency_ms"]].to_numpy() / 1000
    y = (pdf["event_type"] == "buy").to_numpy()
    m = LogisticRegression().fit(X, y)
    return pd.DataFrame({"device": [pdf["device"].iloc[0]], "rows": [len(pdf)],
                         "coef": [float(m.coef_[0, 0])]})

models = (events.filter(F.col("day") >= 23)
          .select("device", "latency_ms", "event_type")
          .groupBy("device")
          .applyInPandas(fit_one, schema="device string, rows long, coef double"))

## Reproduce the numbers quoted in the text

In [5]:
import glob, time
t = time.perf_counter(); fitted = models.collect(); took = time.perf_counter() - t
print(f"applyInPandas: {len(fitted)} models in {took:.1f}s")
for r in fitted:
    print(f"  {r['device']:8s} {r['rows']:,} rows")

need, total, used = {"user_id", "event_type", "amount", "device"}, 0, 0
for f in glob.glob("events/day=*/part-0.parquet"):
    md, day = pq.ParquetFile(f).metadata, int(f.split("day=")[1][:2])
    for g in range(md.num_row_groups):
        for c in range(md.num_columns):
            col = md.row_group(g).column(c)
            total += col.total_compressed_size
            used += col.total_compressed_size if day >= 23 and col.path_in_schema in need else 0
print(f"column chunks the query reads: {used/1e6:.0f} MB of {total/1e6:.0f} MB ({used/total:.1%})")

# The benchmark: one warm-up query, then four with other filters, per engine
import duckdb, polars as pl
cases, warm = [(22, "US"), (24, "JP"), (21, "IN"), (23, "BR")], (23, "DE")
def bench(q):
    t = time.perf_counter(); first = q(*warm); first_s = time.perf_counter() - t
    times = []
    for d, c in cases:
        t = time.perf_counter(); q(d, c); times.append(time.perf_counter() - t)
    return first, first_s, sorted(times)
def show(name, r):
    first, first_s, t = r
    print(f"{name:7s} first {first_s:.2f}s  warm median {(t[1] + t[2]) / 2:.2f}s ({t[0]:.2f} to {t[-1]:.2f})")
    return sorted((row[0], round(row[1], 2), row[2]) for row in first)

con = duckdb.connect()
r_duck = bench(lambda d, c: con.execute(
    "SELECT device, sum(amount), count(*) FROM read_parquet('events/*/*.parquet', hive_partitioning=true) e "
    "JOIN 'users.parquet' u USING (user_id) WHERE CAST(day AS INT) >= ? AND event_type = 'buy' AND country = ? "
    "GROUP BY device", [d, c]).fetchall())
ev_pl, us_pl = pl.scan_parquet("events/*/*.parquet", hive_partitioning=True), pl.scan_parquet("users.parquet")
r_pl = bench(lambda d, c: ev_pl.join(us_pl, on="user_id")
             .filter((pl.col("day").cast(pl.Int32) >= d) & (pl.col("event_type") == "buy") & (pl.col("country") == c))
             .group_by("device").agg(pl.col("amount").sum(), pl.len()).collect().rows())
r_spark = bench(lambda d, c: [tuple(x) for x in events.join(users, "user_id")
                .filter((F.col("day") >= d) & (F.col("event_type") == "buy") & (F.col("country") == c))
                .groupBy("device").agg(F.sum("amount"), F.count("*")).collect()])
answers = [show(n, r) for n, r in [("DuckDB", r_duck), ("Polars", r_pl), ("PySpark", r_spark)]]
print("identical answers to the cent:", answers[0] == answers[1] == answers[2])
print("(timings depend on the machine; the article's ran on a two-core cloud VM with a warm file cache)")

applyInPandas: 3 models in 11.5s
  android  2,332,705 rows
  web      2,333,273 rows
  ios      2,334,022 rows
column chunks the query reads: 53 MB of 1167 MB (4.6%)


DuckDB  first 0.14s  warm median 0.11s (0.10 to 0.14)
Polars  first 0.33s  warm median 0.27s (0.26 to 0.27)
PySpark first 1.89s  warm median 1.02s (0.84 to 1.05)
identical answers to the cent: True
(timings depend on the machine; the article's ran on a two-core cloud VM with a warm file cache)
